# 08 · Reachable corridor against Shapely

**Question.** Does Fieldwork's `reachable_corridor` widget draw the corridor it says it
draws?

The widget declares a rule: for each directed street leaving a reached junction, take the
first `remainingM / lengthM` of its length from that junction, buffer the collected lines
by the half-width, and union them. This rebuilds that from the rule using GEOS through
Shapely, rather than comparing geometry feature by feature.

The fixture is the five-minute band of the Soho corridor — 142 reached junctions, 25 m
either side — and it carries the street geometry, because a corridor cannot be rebuilt
from endpoints: these streets curve and the buffer follows the curve.

**Criteria fixed before running, with reasons.**

1. *Entered and partial segment counts, exactly.* Both implementations select streets by
   the same stated arithmetic over the same numbers, so a difference here would be a
   difference in the rule, not in the geometry engine.
2. *Area within 1%.* Deliberately not tighter. Fieldwork buffers through Turf/JSTS in a
   local azimuthal equidistant projection; this buffers through GEOS in UTM zone 30N. A
   criterion pretending those agree to the micrometre would be measuring the difference
   between map projections and calling it a defect.
3. *Symmetric difference at most 2% of the reported area.* Equal areas can be different
   shapes, so the overlap is checked and not just the total.

In [ ]:
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_08_corridor import run

report = run(Path.cwd() / "fixtures" / "corridor-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"reached junctions: {report['reachedJunctions']}")
print(f"segments entered: Fieldwork {report['fieldwork']['enteredSegments']}, "
      f"Shapely {report['reference']['enteredSegments']}")
print(f"entered part-way: Fieldwork {report['fieldwork']['partialSegments']}, "
      f"Shapely {report['reference']['partialSegments']}")
print(f"area: Fieldwork {report['fieldwork']['areaSquareMetres']:.0f} m2, "
      f"Shapely {report['reference']['areaSquareMetres']:.0f} m2")
print(f"area difference        {report['areaDifferenceFraction']*100:.3f}%")
print(f"symmetric difference   {report['symmetricDifferenceFraction']*100:.3f}%")

## Where the residual comes from

The segment selection agrees exactly: both implementations enter the same 301 streets and
cut 45 of them part-way. That is the part of the operation that carries the method, and it
is reproduced from the written rule rather than copied.

The remaining tenth of a percent is the buffer. Two geometry engines, each approximating a
25 m radius with 8 segments per quadrant, in two different projections. The symmetric
difference being the same size as the area difference tells you it is a boundary effect
spread evenly around the edge, not a piece of corridor present in one and missing from the
other.

## What agreement here does not mean

Both implementations trust the same street geometry and the same lengths, so a wrong
network is wrong in both and this check still passes.

And neither says anything about the half-width. 25 m either side is a claim that anywhere
within 25 m of a street a traveller can enter is reached, which includes the insides of
buildings, locked yards and the far side of walls. No reasoning for 25 m is recorded
anywhere in the case study it came from.

In [ ]:
# How much the corridor's area depends on that unrecorded 25 m, with everything else held
# fixed. The rule is rebuilt at each half-width, so this is the reference implementation
# answering the question, not Fieldwork.
import numpy as np, pyproj, shapely
from shapely.geometry import LineString, MultiLineString
from shapely.ops import substring, unary_union

fixture = json.loads((Path.cwd() / "fixtures" / "corridor-soho-001.json").read_text())
declared = fixture["declaredParameters"]
to_metric = pyproj.Transformer.from_crs("OGC:CRS84", declared["metricCRS"], always_xy=True)
project = lambda g: shapely.transform(
    g, lambda c: np.column_stack(to_metric.transform(c[:, 0], c[:, 1])))

nodes = {n["id"]: n["coordinates"] for n in fixture["input"]["network"]["nodes"]}
remaining = {r["networkNodeId"]: r["remainingM"] for r in fixture["input"]["reached"]}
pieces = []
for edge in fixture["input"]["network"]["edges"]:
    left = remaining.get(edge["from"])
    if left is None or left <= 0:
        continue
    coords = edge.get("geometry") or [nodes[edge["from"]], nodes[edge["to"]]]
    line = project(LineString(coords))
    start = project(shapely.Point(nodes[edge["from"]]))
    if shapely.Point(line.coords[0]).distance(start) > shapely.Point(line.coords[-1]).distance(start):
        line = LineString(list(line.coords)[::-1])
    piece = substring(line, 0.0, min(1.0, left / edge["lengthM"]), normalized=True)
    if not piece.is_empty and piece.length > 0:
        pieces.append(piece)

corridor = lambda half: unary_union(MultiLineString(pieces).buffer(half, quad_segs=8)).area
base = corridor(declared["bufferM"])
print(f"{'half-width m':>13} {'strip m':>8} {'area m2':>10} {'vs 25 m':>9}")
for half in (10, 15, 25, 40, 60):
    area = corridor(half)
    marker = "  <- the case study's figure, with no reason recorded" if half == declared["bufferM"] else ""
    print(f"{half:13d} {half*2:8d} {area:10.0f} {area/base:8.2f}x{marker}")